# IRP Stage 3 — Experiment Tracking
## Scenario 2
**MLOps Group Project | Section 1, Group 5**  
*Maria-Irina Popa, Enzo Jerez, Roberto Cummings, Jia Yi Rachel Lee, Thomas Christian Matenco*

---

run this in the terminal:

mlflow server --backend-store-uri sqlite:///backend.db --default-artifact-root ./mlartifacts --host 127.0.0.1 --port 5000

In [2]:
import os
import json
import warnings
import subprocess
import sys
from pathlib import Path

import mlflow
import mlflow.sklearn
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
print("Libraries loaded")


Libraries loaded


In [3]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("inventory-risk-stage3-team")

2026/03/25 03:07:33 INFO mlflow.tracking.fluent: Experiment with name 'inventory-risk-stage3-team' does not exist. Creating a new experiment.


<Experiment: artifact_location=('/Users/ljyr/Documents/IE/Term 2/1 MLOPS Machine Learning '
 'Operations/IRP-Section1Group5/03-experiment-tracking/mlartifacts/1'), creation_time=1774404453086, experiment_id='1', last_update_time=1774404453086, lifecycle_stage='active', name='inventory-risk-stage3-team', tags={}, workspace='default'>

In [4]:
train = pd.read_parquet("../02-features-modelling/data/train.parquet")
val   = pd.read_parquet("../02-features-modelling/data/val.parquet")
test  = pd.read_parquet("../02-features-modelling/data/test.parquet")

print(train.shape, val.shape, test.shape)
print(train.head(2))

(53900, 30) (12300, 30) (6000, 30)
        Date Store ID Product ID     Category Region  Inventory Level  \
0 2022-01-08     S001      P0001    Furniture   East              231   
1 2022-01-09     S001      P0001  Electronics  South              373   

   Units Sold  Units Ordered  Demand Forecast  Price  ...  Inventory_Change  \
0           2            119             0.84  66.30  ...              -1.0   
1         350            178           352.24  41.72  ...             117.0   

  Inventory_Change_Pct  Days_of_Stock  Inventory_vs_Rolling7 Sales_Velocity  \
0            -0.001965     254.000000             173.857143       0.003937   
1             0.230315       1.785714             251.285714       0.560000   

   Coverage_Ratio  Forecast_Error  Order_to_Inventory  Risk_Label_Current  \
0      508.000000            1.16            0.234252           Safe Zone   
1        1.774358           -2.24            0.284800           Safe Zone   

   Risk_Label  
0   Safe Zone  
1   S

## Recreate the Stage 2 Feature setup

In [5]:
TARGET = "Risk_Label"

# Same leakage / non-model columns used in Stage 2
DROP_COLS = [
    "Risk_Label",
    "Risk_Label_Current",
    "Store ID",
    "Product ID",
    "Date",
    "Demand Forecast",
    "Demand_Forecast_Clean",
]

X_train = train.drop(columns=DROP_COLS).copy()
X_val = val.drop(columns=DROP_COLS).copy()
X_test = test.drop(columns=DROP_COLS).copy()

y_train_raw = train[TARGET].copy()
y_val_raw = val[TARGET].copy()
y_test_raw = test[TARGET].copy()

numerical_features = [
    "Inventory_Reconstructed",
    "Units Sold",
    "Units Ordered",
    "Price",
    "Discount",
    "Units_Sold_Lag1",
    "Inventory_Change_Pct",
    "Days_of_Stock",
    "Sales_Velocity",
    "Coverage_Ratio",
    "Forecast_Error",
    "Order_to_Inventory",
]

categorical_features = [
    "Category",
    "Region",
    "Weather Condition",
    "Seasonality",
]

print("X_train shape:", X_train.shape)
print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)


X_train shape: (53900, 23)
Numerical features: ['Inventory_Reconstructed', 'Units Sold', 'Units Ordered', 'Price', 'Discount', 'Units_Sold_Lag1', 'Inventory_Change_Pct', 'Days_of_Stock', 'Sales_Velocity', 'Coverage_Ratio', 'Forecast_Error', 'Order_to_Inventory']
Categorical features: ['Category', 'Region', 'Weather Condition', 'Seasonality']


In [6]:
# Match Stage 2 target encoding flow
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_val = le.transform(y_val_raw)
y_test = le.transform(y_test_raw)

class_names = list(le.classes_)
all_labels = list(range(len(class_names)))

print("Encoded class order:", class_names)


Encoded class order: ['Overstock Risk', 'Safe Zone', 'Stockout Risk']


## Build the same 3 models from Stage 2

In [7]:
# Logistic Regression pipeline
logit_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
])

logit_pipeline = Pipeline([
    ("preprocessor", logit_preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42,
    )),
])

# Random Forest pipeline
X_train_rf = X_train.copy()
X_val_rf = X_val.copy()
X_test_rf = X_test.copy()
for col in categorical_features:
    X_train_rf[col] = X_train_rf[col].astype("object")
    X_val_rf[col] = X_val_rf[col].astype("object")
    X_test_rf[col] = X_test_rf[col].astype("object")

cat_indices = [X_train_rf.columns.get_loc(col) for col in categorical_features]
smote_nc = SMOTENC(categorical_features=cat_indices, random_state=42)

rf_preprocessor = ColumnTransformer([
    ("num", "passthrough", numerical_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

rf_pipeline = Pipeline([
    ("smote", smote_nc),
    ("preprocessor", rf_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=10,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    )),
])

# XGBoost pipeline
xgb_preprocessor = ColumnTransformer([
    ("num", "passthrough", numerical_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

xgb_pipeline = SkPipeline([
    ("preprocessor", xgb_preprocessor),
    ("model", XGBClassifier(
        objective="multi:softmax",
        num_class=3,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="mlogloss",
    )),
])

classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))
sample_weights = np.array([class_weight_dict[y] for y in y_train])

print("Pipelines ready")
print("XGBoost class weights:", class_weight_dict)


Pipelines ready
XGBoost class weights: {np.int64(0): np.float64(5.459333535906007), np.int64(1): np.float64(0.8282241583306443), np.int64(2): np.float64(0.6213399732558675)}


## Training + evaluation helper

In [8]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
    }


def save_confusion_matrix(y_true, y_pred, split_name, model_name, out_dir):
    cm = confusion_matrix(y_true, y_pred, labels=all_labels)
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    csv_path = out_dir / f"{split_name.lower()}_confusion_matrix.csv"
    fig_path = out_dir / f"{split_name.lower()}_confusion_matrix.png"

    cm_df.to_csv(csv_path)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{model_name} - {split_name} confusion matrix")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(fig_path, dpi=150)
    plt.close()


def save_classification_report(y_true, y_pred, split_name, out_dir):
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=all_labels,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report_dict).transpose()
    report_df.to_csv(out_dir / f"{split_name.lower()}_classification_report.csv")


def fit_and_predict(model_name, model):
    if model_name == "Random Forest":
        model.fit(X_train_rf, y_train)
        return {
            "train": model.predict(X_train_rf),
            "val": model.predict(X_val_rf),
            "test": model.predict(X_test_rf),
        }
    if model_name == "XGBoost":
        model.fit(X_train, y_train, model__sample_weight=sample_weights)
        return {
            "train": model.predict(X_train),
            "val": model.predict(X_val),
            "test": model.predict(X_test),
        }

    model.fit(X_train, y_train)
    return {
        "train": model.predict(X_train),
        "val": model.predict(X_val),
        "test": model.predict(X_test),
    }


## Run MLFlow experiments

In [9]:
models = {
    "Logistic Regression": logit_pipeline,
    "Random Forest": rf_pipeline,
    "XGBoost": xgb_pipeline,
}

results = []
artifacts_root = Path("mlflow_artifacts")
artifacts_root.mkdir(exist_ok=True)

for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name) as run:
        run_id = run.info.run_id
        run_dir = artifacts_root / run_id
        run_dir.mkdir(parents=True, exist_ok=True)

        mlflow.set_tag("scenario", "local-only")
        mlflow.set_tag("stage", "stage3")
        mlflow.set_tag("model_type", model_name)
        mlflow.set_tag("project", "inventory-risk")

        mlflow.log_param("sklearn_version", sklearn.__version__)
        mlflow.log_param("train_rows", len(train))
        mlflow.log_param("val_rows", len(val))
        mlflow.log_param("test_rows", len(test))
        mlflow.log_param("n_numerical_features", len(numerical_features))
        mlflow.log_param("n_categorical_features", len(categorical_features))
        mlflow.log_param("target_name", TARGET)
        mlflow.log_param("class_names", json.dumps(class_names))

        if model_name == "Logistic Regression":
            mlflow.log_params({
                "max_iter": 2000,
                "class_weight": "balanced",
                "smote": True,
            })
        elif model_name == "Random Forest":
            mlflow.log_params({
                "n_estimators": 300,
                "max_depth": 10,
                "min_samples_leaf": 10,
                "max_features": "sqrt",
                "smotenc": True,
            })
        elif model_name == "XGBoost":
            mlflow.log_params({
                "objective": "multi:softmax",
                "num_class": 3,
                "n_estimators": 300,
                "max_depth": 6,
                "learning_rate": 0.05,
                "subsample": 0.8,
                "colsample_bytree": 0.8,
                "sample_weighting": "balanced",
            })

        preds = fit_and_predict(model_name, model)

        split_targets = {
            "train": y_train,
            "val": y_val,
            "test": y_test,
        }

        summary = {"model": model_name, "run_id": run_id}

        for split_name, y_true in split_targets.items():
            y_pred = preds[split_name]
            metrics = compute_metrics(y_true, y_pred)
            for metric_name, metric_value in metrics.items():
                mlflow.log_metric(f"{split_name}_{metric_name}", metric_value)
                summary[f"{split_name}_{metric_name}"] = round(metric_value, 4)

            save_confusion_matrix(y_true, y_pred, split_name.upper(), model_name, run_dir)
            save_classification_report(y_true, y_pred, split_name.upper(), run_dir)

        mlflow.log_artifacts(str(run_dir), artifact_path="reports")
        mlflow.sklearn.log_model(model, artifact_path="model")

        results.append(summary)
        print(f"Logged {model_name} | run_id={run_id} | val_f1_macro={summary['val_f1_macro']:.4f}")

results_df = pd.DataFrame(results).sort_values(by="val_f1_macro", ascending=False).reset_index(drop=True)
results_df


2026/03/25 03:07:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 03:07:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged Logistic Regression | run_id=96f6b54850f9451aabe74e5eb98d5bcd | val_f1_macro=0.5088
🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/1/runs/96f6b54850f9451aabe74e5eb98d5bcd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/03/25 03:08:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 03:08:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged Random Forest | run_id=2652a60fc90d486e905c448df3c68d0d | val_f1_macro=0.5243
🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/1/runs/2652a60fc90d486e905c448df3c68d0d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/03/25 03:08:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 03:08:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged XGBoost | run_id=b4e7785ad45a4a3e81f72cefddce0574 | val_f1_macro=0.5427
🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/1/runs/b4e7785ad45a4a3e81f72cefddce0574
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


,model,run_id,train_accuracy,train_f1_macro,train_precision_macro,train_recall_macro,val_accuracy,val_f1_macro,val_precision_macro,val_recall_macro,test_accuracy,test_f1_macro,test_precision_macro,test_recall_macro
0,XGBoost,b4e7785ad45a4a3e81f72cefddce0574,0.7871,0.7198,0.6943,0.8253,0.6957,0.5427,0.5498,0.5576,0.7003,0.5451,0.5530,0.5596
1,Random Forest,2652a60fc90d486e905c448df3c68d0d,0.6842,0.5642,0.5822,0.6237,0.6629,0.5243,0.5449,0.5610,0.6578,0.5231,0.5470,0.5694
2,Logistic Regression,96f6b54850f9451aabe74e5eb98d5bcd,0.6314,0.5032,0.5476,0.5717,0.6465,0.5088,0.5477,0.5723,0.6387,0.4965,0.5398,0.5573


## Save best run for Stage 4

In [10]:
best_row = results_df.iloc[0]
best_run_id = best_row["run_id"]
best_model_name = best_row["model"]

with open("run_id.txt", "w") as f:
    f.write(best_run_id)

print("Best model:", best_model_name)
print("Best run_id:", best_run_id)
print("Saved to run_id.txt")


Best model: XGBoost
Best run_id: b4e7785ad45a4a3e81f72cefddce0574
Saved to run_id.txt
